# Machine Assisted Techniques

This notebook provides the methods used for machine assisted techniques (Text Classification, Sentiment Analysis, Topic Analysis)

---

## Download and Import Dependencies

In [2]:
from transformers.pipelines.pt_utils import KeyDataset
from transformers import AutoModelForSequenceClassification, TextClassificationPipeline, AutoTokenizer, AutoConfig, pipeline
from google.colab import drive
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

# Imports for BERTopic
!pip install bertopic
!pip install sentence_transformers
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from umap import UMAP
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
import pickle

drive.mount('/content/drive')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Mounted at /content/drive


## Text Classification

This technique utilizes the custom model trained for this analysis

In [3]:
data_path = "/content/drive/MyDrive/Bosnian Parliment/segmented_BiHCorp.parquet"
raw_dataset = load_dataset('parquet', data_files=data_path, split='train') # Train needs to be used even though there is no split

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
# Load the Model & Data
model_id = "zastuck/roberta-base-bosnian-parliament-multilabel-v1" # Load model from HuggingFace
classifier = pipeline("text-classification", model=model_id, device=0, batch_size=32, top_k=None)

id2label = classifier.model.config.id2label
class_names = [id2label[i] for i in range(len(id2label))]

#  Generate Raw Probabilities
raw_scores = []
for out in tqdm(classifier(KeyDataset(raw_dataset, "segment")), total=len(raw_dataset)):
    # 'out' is a list: [{'label': 'PROCEDURAL', 'score': 0.9}, {'label': 'COLLECT_MEM', 'score': 0.1}, ...]
    # turns this into a dictionary: {'PROCEDURAL': 0.9, 'COLLECT_MEM': 0.1, ...}
    label_dict = {item['label']: item['score'] for item in out}
    raw_scores.append(label_dict)

# Creating the DataFrame from a list of dicts
df_results = pd.DataFrame(raw_scores)
for col in class_names:
    df_results[f"{col}_bool"] = df_results[col] >= 0.5

# Add the original text and Id
df_full = pd.concat([raw_dataset.to_pandas(), df_results], axis=1)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  0%|          | 0/120783 [00:00<?, ?it/s]

## Sentiment Analysis

In [5]:
# Initialize model based on documentation of model
MODEL = "classla/xlm-r-parlasent"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

# Initialize pipeline
sent_pipe = TextClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
    task='sentiment_analysis',
    device=0,
    function_to_apply="none" # Keeping raw logits as per ParlaSent's original spec
)

# Define map function
def compute_sentiment(batch):
    texts = [str(t) if (t is not None and len(str(t).strip()) > 0) else "Neutral." for t in batch["segment"]]

    # Pass the list directly to the pipe
    outputs = sent_pipe(texts, batch_size=32) # Define batching here

    scores = []
    for out in outputs:
        scores.append(out['score'])

    return {"sentiment_results": scores}

# Map to dataset
raw_dataset = raw_dataset.map(compute_sentiment, batched=True, batch_size=32)

# Concat with full DataFrame
df_full = pd.concat([df_full, raw_dataset.to_pandas()['sentiment_results']], axis=1)
df_full.to_parquet("/content/drive/MyDrive/Bosnian Parliment/segmented_BiHCorp_class_sent.parquet", compression="zstd")

config.json:   0%|          | 0.00/876 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Map:   0%|          | 0/120783 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


AttributeError: 'Column' object has no attribute 'to_pandas'

## Topic Analysis

Only segments classified as SUBSTANTIVE_IDENTITY used to prevent the model clustering topics that are purely procedural or not relevant to the analysis

In [11]:
df_topic = df_full[df_full["SUBSTANTIVE_IDENTITY_bool"] == True] # Filter for only substantive

# Remove polite introductions to avoid modelling as a topic
patterns = [
    r"thank you (mr|madam) (chairman|speaker|president)",
    r"i yield my time",
    r"i move that the bill",
]

combined_pattern = "|".join(patterns)

# Apply to the column directly
df_topic['segment_clean'] = df_topic['segment'].str.replace(combined_pattern, "", case=False, regex=True).str.strip()
texts = df_topic['segment_clean']

### Creating Embeddings and BERTopic Model

The embeddings and model were saved for replication and validity of results. This chunk includes all parameters used for BERTopic

In [13]:
default_stopwords = stopwords.words("english") # Import stopwords list from NLTK
my_stopwords = ["chairman", "speaker", "floor", "colleagues", "thank", "mr", "madam"] # Reduce chance of model clustering parliamentary norms
default_stopwords.extend(my_stopwords)
vectorizer_model = CountVectorizer(stop_words=default_stopwords)

# Create embeddings before hand for replication
sentence_model = SentenceTransformer("all-mpnet-base-v2") # Provides detailed embeddings
embeddings = sentence_model.encode(texts, show_progress_bar=False)
np.save('/content/drive/MyDrive/Bosnian Parliment/BERTopic/embeddings.npy', embeddings) # Save for replication


representation_model = KeyBERTInspired() # Used for better interpretability (see BERTopic documentation)

# Define models with fixed seeds
umap_model = UMAP(
    n_neighbors=20,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)
hdbscan_model = HDBSCAN(
    min_cluster_size=60,       # Smaller topics allowed (approx. 0.1% of corpus)
    min_samples=10,            # Lower value = fewer outliers/noise
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# Pass models used to BERTopic
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    representation_model=representation_model,
    vectorizer_model=vectorizer_model,
    embedding_model = sentence_model
    )

topics, probs = topic_model.fit_transform(texts, embeddings)

# Reduce outliers using c-TF-IDF, distributions, and embeddings (strengthens final results)
new_topics = topic_model.reduce_outliers(documents=texts, topics=topics, strategy="c-TF-IDF", threshold=0.1)
new_topics = topic_model.reduce_outliers(documents=texts, topics=new_topics, strategy="distributions")
new_topics = topic_model.reduce_outliers(documents=texts, topics=new_topics, strategy="embeddings", embeddings=embeddings)
topic_model.update_topics(docs=texts, topics=new_topics, vectorizer_model=vectorizer_model)

topic_model.save("/content/drive/MyDrive/Bosnian Parliment/BERTopic/BERTopic-bosnian-parliament", serialization="pickle") # Save for replication

KeyboardInterrupt: 

### Modeling from pre-created model
All embeddings and the topic model were saved for replication. Use this for constent results

In [15]:
embeddings = np.load('/content/drive/MyDrive/Bosnian Parliment/BERTopic/embeddings.npy') # Load existing embeddings

topic_model = BERTopic.load('/content/drive/MyDrive/Bosnian Parliment/BERTopic/BERTopic-bosnian-parliament') # Load existing BERTopic model

topics, probs = topic_model.fit_transform(texts, embeddings) # Map the model to the texts and embeddings

In [16]:
df_topic["topic_id"] = topics # Add the topics to the DataFrame

# Map the topic names (e.g., "0_healthcare_hospitals") to the IDs
topic_info = topic_model.get_topic_info()
topic_mapping = dict(zip(topic_info.Topic, topic_info.Name))

df_topic['topic_label'] = df_topic['topic_id'].map(topic_mapping).fillna("Other")
df_topic.drop(columns=["segment_clean"], inplace=True)

df_topic.to_parquet("/content/drive/MyDrive/Bosnian Parliment/segmented_BiHCorp_final.parquet", compression="zstd") # Export final data for analysis

,segment_id,doc_id,house,term,date,meeting,agenda,fullname,party,segment,...,SELF_ETHNIC_bool,SELF_GOV_bool,SELF_PARTY_bool,SELF_STATE_bool,SELF_TRANS_bool,SUBSTANTIVE_IDENTITY_bool,SUBSTANTIVE_TECHNICAL_bool,sentiment_results,topic_id,topic_label
9,10,17323,DN,2006-2010,2010-01-21,42. sjednica,intro,"Majkić, Dušanka",SNSD,. Since a delegate may be an authorized propos...,...,False,True,False,False,False,True,False,0.189775,0,0_law_constitutional_procedure_laws
10,11,17323,DN,2006-2010,2010-01-21,42. sjednica,intro,"Majkić, Dušanka",SNSD,". So, in order not to discuss this, I suggest ...",...,False,False,False,False,False,True,False,0.142858,0,0_law_constitutional_procedure_laws
17,18,17333,DN,2006-2010,2010-01-21,42. sjednica,intro,"Majkić, Dušanka",SNSD,". So, I think it'd be more right to do that be...",...,False,False,False,False,False,True,False,0.106183,0,0_law_constitutional_procedure_laws
18,19,17337,DN,2006-2010,2010-01-21,42. sjednica,intro,"Ljubić, Božo",HDZ 1990,"Dear Presidency, colleagues and fellow envoys,...",...,False,True,False,False,False,True,False,4.445660,0,0_law_constitutional_procedure_laws
21,22,17347,DN,2006-2010,2010-01-21,42. sjednica,intro,"Majkić, Dušanka",SNSD,"It takes six months for the same law. Well, do...",...,False,True,False,False,False,True,False,0.304469,0,0_law_constitutional_procedure_laws
